In [ ]:
import json
import pandas as pd
import ast
from peft import LoraConfig, get_peft_model
from transformers import DataCollatorForLanguageModeling, AutoModelForCausalLM
import re
from tqdm import tqdm
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset

n_conversations = "movie-corpus/conversations.json"
n_utterances = "movie-corpus/utterances.jsonl"
lst_genres = ['action', 'adventure', 'animation', 'biography', 'comedy', 'crime', 'documentary', 'drama', 'family', 'fantasy', 'film-noir', 'horror', 'mystery', 'romance', 'sci-fi', 'short', 'thriller']

In [27]:
with open(n_conversations, "r", encoding="utf-8") as f:
    conversations = json.load(f)

movies = []

for conv in conversations.values():
    meta = conv["meta"]
    genres = ast.literal_eval(meta["genre"])

    if not genres:
        continue
    movies.append({"movie_id": meta["movie_idx"], "movie_name": meta["movie_name"], "genre": genres[0]})

movies_df = pd.DataFrame(movies).drop_duplicates("movie_id")


In [28]:
utterances = []

with open(n_utterances, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        utterances.append({"conversation_id": data["conversation_id"], "text": data["text"], "movie_id": data["meta"]["movie_id"]
        })

utterances_df = pd.DataFrame(utterances)
dialogues_df = (
    utterances_df
    .groupby(["conversation_id", "movie_id"])["text"]
    .apply(lambda x: " ".join(x))
    .reset_index()
    .rename(columns={"text": "dialogue"})
)

full_df = dialogues_df.merge(
    movies_df,
    on="movie_id",
    how="inner"
)

In [ ]:
def build_prompt(row):
    genres_str = ", ".join(lst_genres)
    return f"""Dialogue:
{row.dialogue}

Question:
What is the genre of this movie?
Choose one from those :{genres_str}

Answer:
{row.genre}
"""


full_df["text"] = full_df.apply(build_prompt, axis=1)

train_df, test_df = train_test_split(full_df, test_size=0.2, random_state=42, stratify=full_df["genre"])
train_dataset = Dataset.from_pandas(train_df[["text"]])
test_dataset = Dataset.from_pandas(test_df[["text"]])

In [31]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256
        #max_length=512
    )

train_dataset = train_dataset.map(tokenize, batched=True, remove_columns=["text"])
test_dataset = test_dataset.map(tokenize, batched=True, remove_columns=["text"])

Map: 100%|██████████| 16598/16598 [00:01<00:00, 13293.77 examples/s]


In [ ]:
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],
    #lora_dropout=0.05,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


c:\Users\paull\AppData\Local\Programs\Python\Python310\lib\site-packages\peft\tuners\lora\layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt-genre",
    #per_device_train_batch_size=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    #num_train_epochs=3,
    fp16=True,
    logging_steps=50,
    #save_strategy="epoch"
    save_strategy="no"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, data_collator=data_collator)

trainer.train()

Step,Training Loss
50,3.747700
100,3.532700
150,3.259900
200,2.936900
250,2.701200
300,2.516900
350,2.366700
400,2.289000
450,2.176100
500,2.133300


TrainOutput(global_step=8300, training_loss=1.848601377556123, metrics={'train_runtime': 2112.7445, 'train_samples_per_second': 62.845, 'train_steps_per_second': 3.929, 'total_flos': 8703557096177664.0, 'train_loss': 1.848601377556123, 'epoch': 2.0})

In [34]:
import torch

prompt = """Dialogue:
I cann't do it anymore!

Question:
What is the genre of this movie?

Answer:
"""

device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model.generate(
    **inputs,
    max_new_tokens=5,
    do_sample=False
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Dialogue:
I cann't do it anymore!

Question:
What is the genre of this movie?

Answer:
comedy, adventure,


In [ ]:
GENRE_TO_IDX = {g: i for i, g in enumerate(lst_genres)}
IDX_TO_GENRE = {i: g for g, i in GENRE_TO_IDX.items()}

def parse_predicted_genre(text):
    text = text.lower()
    text = re.sub(r"[^a-z,\-\s]", "", text)
    for g in lst_genres:
        if g in text:
            return GENRE_TO_IDX[g]
    return None

def true_genre_to_label(genre):
    if genre is None:
        return None
    return GENRE_TO_IDX.get(genre)

In [38]:
y_true = []
y_pred = []

model.eval()

for row in tqdm(test_df.itertuples(), total=len(test_df)):

    true_label = GENRE_TO_IDX.get(row.genre)
    if true_label is None:
        continue

    prompt = build_prompt(row)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=3,
            do_sample=False
        )

    generated = tokenizer.decode(output[0], skip_special_tokens=True)

    pred_label = parse_predicted_genre(generated)
    if pred_label is None:
        continue

    y_true.append(true_label)
    y_pred.append(pred_label)

  0%|          | 1/16598 [00:00<56:14,  4.92it/s]Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
  0%|          | 7/16598 [00:00<10:09, 27.23it/s]Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
  0%|          | 13/16598 [00:00<07:06